In [13]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("../data/fda_drug_recalls.xls")

print(f"Data Loaded — {df.shape[0]:,} records | {df.shape[1]} columns")

Data Loaded — 17,529 records | 25 columns


In [14]:
# Drop columns that are mostly empty or not needed for analysis
cols_to_drop = ['more_code_info', 'address_2', 'code_info', 
                'openfda', 'address_1', 'postal_code']

df = df.drop(columns=cols_to_drop)

print(f"Columns remaining : {df.shape[1]}")
print(f"Columns dropped   : {len(cols_to_drop)}")
print("\nRemaining columns:")
print(df.columns.tolist())

Columns remaining : 19
Columns dropped   : 6

Remaining columns:
['status', 'city', 'state', 'country', 'classification', 'product_type', 'event_id', 'recalling_firm', 'voluntary_mandated', 'initial_firm_notification', 'distribution_pattern', 'recall_number', 'product_description', 'product_quantity', 'reason_for_recall', 'recall_initiation_date', 'center_classification_date', 'report_date', 'termination_date']


In [15]:
# Convert all date columns to proper datetime format
date_cols = ['recall_initiation_date', 'termination_date', 
             'center_classification_date', 'report_date']

for col in date_cols:
    df[col] = pd.to_datetime(df[col], format='%Y%m%d', errors='coerce')

print("Date columns converted")
print(df[date_cols].dtypes)

Date columns converted
recall_initiation_date        datetime64[ns]
termination_date              datetime64[ns]
center_classification_date    datetime64[ns]
report_date                   datetime64[ns]
dtype: object


In [16]:
# This is our KEY metric for A/B testing
df['resolution_days'] = (
    df['termination_date'] - df['recall_initiation_date']
).dt.days

print(" Resolution days column created")
print(f"\nAverage resolution time : {df['resolution_days'].mean():.0f} days")
print(f"Minimum resolution time : {df['resolution_days'].min():.0f} days")
print(f"Maximum resolution time : {df['resolution_days'].max():.0f} days")
print(f"\nMissing (still ongoing) : {df['resolution_days'].isnull().sum():,} records")

 Resolution days column created

Average resolution time : 673 days
Minimum resolution time : 8 days
Maximum resolution time : 4247 days

Missing (still ongoing) : 2,751 records


In [20]:
# Drop rows where voluntary_mandated is missing
df = df.dropna(subset=['voluntary_mandated', 'classification'])

# For A/B testing create a clean dataframe 
# with only resolved recalls
df_resolved = df[df['resolution_days'].notna()].copy()

print(f"Total records         : {len(df):,}")
print(f"Resolved records      : {len(df_resolved):,}")
print(f"Ongoing records       : {len(df) - len(df_resolved):,}")
print(f"\ndf_resolved is ready for A/B Testing")

Total records         : 17,494
Resolved records      : 14,746
Ongoing records       : 2,748

df_resolved is ready for A/B Testing


In [21]:
# Save full cleaned dataset
df.to_csv("../data/fda_cleaned.csv", index=False)

# Save resolved only dataset for A/B testing
df_resolved.to_csv("../data/fda_resolved.csv", index=False)

print("✅ fda_cleaned.csv saved")
print("✅ fda_resolved.csv saved")
print(f"\nfda_cleaned   : {df.shape}")
print(f"fda_resolved  : {df_resolved.shape}")

✅ fda_cleaned.csv saved
✅ fda_resolved.csv saved

fda_cleaned   : (17494, 20)
fda_resolved  : (14746, 20)
